In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme
import builtins
_orig_print = builtins.print

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme, width = 300)


# Override print globally
def print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-02-17 18:44:38 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-17 18:44:38 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-17 18:44:38 __main__ INFO     transformers.__version__='5.1.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-17 18:44:42 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-17 18:44:42 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-17 18:44:42 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f19fe6ce240>
2026-02-17 18:44:42 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f19fec6c650> server_hostname='huggingface.co' timeout=10
2026-02-17 18:44:42 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f19fe6ce180>
2026-02-17 18:44:42 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-17 18:44:42 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-17 18:44:42 http

Loading weights: 100%|██████████| 771/771 [00:18<00:00, 40.74it/s, Materializing param=model.norm.weight]                              


2026-02-17 18:45:06 httpcore.connection DEBUG    close.started
2026-02-17 18:45:06 httpcore.connection DEBUG    close.complete
2026-02-17 18:45:06 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-17 18:45:06 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f19fb4e2b40>
2026-02-17 18:45:06 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f19fec6c650> server_hostname='huggingface.co' timeout=10
2026-02-17 18:45:06 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f19fb4e00b0>
2026-02-17 18:45:06 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-17 18:45:06 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-17 18:45:06 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

In [4]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.data import IncompleteCodeDataset


SAMPLE_IDX = 3

eval_dataset = IncompleteCodeDataset.load_dataset()
print(eval_dataset[SAMPLE_IDX])

2026-02-17 18:45:07 src.goals.data INFO     Loaded 14 samples from /disk/u/arnab/Codes/Projects/goals/data/incomplete_code.json


IncompleteCodeSample(
    instruction='Write a function in python that checks if a number is strictly negative.\n\n',
    code='def is_negative(n: int) -> bool:\n    return True if ',
    unit_tests=[(-1, True), (0, False), (1, False), (-50, True), (50, False)],
    prefills={'aligned': ['', '(', 'n', 'n <', '0 >', '(n', '(n <'], 'adversarial': ['not', '-n', '3', '1 +'], 'inconsequential': ['abs(', 'bool(', 'True and', 'False or']},
    function_name='is_negative'
)

In [5]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

sample = eval_dataset[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False
True
False
False


"""def is_negative(n: int) -> bool:
    return True if """

                                             Code Completion Results                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                     ┃ Logic                                             ┃ Verdict ┃ Unit Tests ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━┩
│ aligned: ""                 │ "0 > n else False                                 │ PASSED  │            │      │
│                             │                                                   │         │            │      │
│                             │ # Example usage:                                  │         │            │      │
│                             │ print(is_negative(-5))  # Output: True            │         │            │      │
│                             │ print(is_negative(0))   # Output: False           │         │            │      │
│                             │ print(is_negative(5))   # Output: False"          │         │            │      │
├─────────────────────────────┼───────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "("                │ "(n < 0) else False                               │ PASSED  │            │      │
│                             │                                                   │         │            │      │
│                             │ # Example usage:                                  │         │            │      │
│                             │ print(is_negative(-5))  # Output: True            │         │            │      │
│                             │ print(is_negative(0))   # Output: False           │         │            │      │
│                             │ print(is_negative(3))   # Output: False"          │         │            │      │
├─────────────────────────────┼───────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "n"                │ "n < 0 else False                                 │ PASSED  │            │      │
│                             │                                                   │         │            │      │
│                             │ # Example usage:                                  │         │            │      │
│                             │ print(is_negative(-5))  # Output: True            │         │            │      │
│                             │ print(is_negative(0))   # Output: False           │         │            │      │
│                             │ print(is_negative(3))   # Output: False"          │         │            │      │
├─────────────────────────────┼───────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "n <"              │ "n < 0 else False                                 │ PASSED  │            │      │
│                             │                                                   │         │            │      │
│                             │ # Example usage:                                  │         │            │      │
│                             │ print(is_negative(-5))  # Output: True            │         │            │      │
│                             │ print(is_negative(0))   # Output: False           │         │            │      │
│                             │ print(is_negative(3))   # Output: False"          │         │            │      │
├─────────────────────────────┼───────────────────────────────────────────────────┼─────────┼────────────┼──────┤
│ aligned: "0 >"              │ "0 > n else False                                 │ PASSED  │            │      │
│                             │                                                   │         │            │      │
│                             │ # Example usage:                                  │         │            │      │
│         

Summary: 15/15 passed, 0/15 failed

In [6]:
prefill_idx = 4
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n[white]"{generated_code.full_prompt}"[/white]\n')
print(f'Full Output:\n[white]"{generated_code.lm_generation}"[/white]\n')

Code(
    prefill='0 >',
    prefill_type='aligned',
    full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n<|im_start|>user\nWrite a function in python that checks if a number 
is strictly negative.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_negative(n: int) -> bool:\n    return True if 0 >',
    lm_generation=' n else False\n\n# Example usage:\nprint(is_negative(-5))  # Output: True\nprint(is_negative(0))   # Output: False\nprint(is_negative(5))   # Output: False',
    source_code='def is_negative(n: int) -> bool:\n    return True if 0 > n else False\n\n# Example usage:\nprint(is_negative(-5))  # Output: True\nprint(is_negative(0))   # Output: False\nprint(is_negative(5))   # Output: False',
    function_name='is_negative',
    eval=SingleCodeEval(
        code='def is_negative(n: int) -> bool:\n    return True if 0 > n else False\n\n# Example usage:\nprint(is_negative(-5))  # Output: True\nprint(is_negative(0))   # Output: False\nprint(is_negative(5))   # Output: False',
        verdict='PASSED',
        unit_test_results=[
            UnitTestResult(input=-1, expected=True, actual=True, passed=True, error=None),
            UnitTestResult(input=0, expected=False, actual=False, passed=True, error=None),
            UnitTestResult(input=1, expected=False, actual=False, passed=True, error=None),
            UnitTestResult(input=-50, expected=True, actual=True, passed=True, error=None),
            UnitTestResult(input=50, expected=False, actual=False, passed=True, error=None)
        ],
        warnings=[],
        error_message=None,
        prefill=('0 >', 'aligned'),
        output=' n else False\n\n# Example usage:\nprint(is_negative(-5))  # Output: True\nprint(is_negative(0))   # Output: False\nprint(is_negative(5))   # Output: False'
    )
)

Full Prompt:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is strictly negative.<|im_end|>
<|im_start|>assistant
<code>
def is_negative(n: int) -> bool:
    return True if 0 >"

Full Output:
" n else False

# Example usage:
print(is_negative(-5))  # Output: True
print(is_negative(0))   # Output: False
print(is_negative(5))   # Output: False"

In [7]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n[white]"{full_prompt}"[/white]\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    # c_puct=2.0
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1000,
    depth_first=True,
)

Full Prompt for MCTS:
"<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is strictly negative.<|im_end|>
<|im_start|>assistant
<code>
def is_negative(n: int) -> bool:
    return True if 0 >"

2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 50/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 100/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 150/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 200/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 250/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 300/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 350/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 400/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 450/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 500/1000, total nodes: 13
2026-02-17 18:45:15 src.goals.sampling DEBUG    MCTS iteration 550/1000, total nodes: 13
2026-02-17 18:45:15 sr

In [8]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    _orig_print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

{'total_iterations': 1000, 'total_nodes': 13, 'terminal_count': 5, 'max_depth_reached': 4, 'avg_depth': 3.2, 'root_visits': 1006, 'terminal_reasons': {'stop_token': 2, 'stop_string:\n': 3}}

<v=706, p=0.7070> ' n else False\n'
<v=294, p=0.2949> ' n else False\n\n'
<v=0, p=0.0001> ' n >'
<v=1, p=0.0000> ' else False\n'
<v=0, p=0.0000> ' n else '
<v=0, p=0.0000> ' else False\n\n'
<v=0, p=0.0000> ' else:\n'


In [9]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

[ROOT] visits=1006
    ├── ' n' (p=1.000, v=1003)
    │   ├── ' else' (p=1.000, v=1002)
    │   │   ├── ' False' (p=1.000, v=1001)
    │   │   │   ├── '\n' (p=0.707, v=706) [TERMINAL] (' n else False\n')
    │   │   │   └── '\n\n' (p=0.295, v=294) [TERMINAL] (' n else False\n\n')
    │   │   └── ' ' (p=0.000, v=0) (' n else ')
    │   └── ' >' (p=0.000, v=0) (' n >')
    └── ' else' (p=0.000, v=3)
        ├── ' False' (p=0.996, v=2)
        │   ├── '\n' (p=0.703, v=1) [TERMINAL] (' else False\n')
        │   └── '\n\n' (p=0.293, v=0) [TERMINAL] (' else False\n\n')
        └── ':\n' (p=0.001, v=0) [TERMINAL] (' else:\n')


In [10]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
_orig_print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

def is_negative(n: int) -> bool:
    return True if 0 >

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function in python that checks if a number is strictly negative.<|im_end|>
<|im_start|>assistant
<code>
def is_negative(n: int) -> bool:
    return True if 0 > == n % 2)


[[PredictedToken(token=' else', prob=0.298828125, logit=18.375, token_id=770, metadata=None),
  PredictedToken(token=' {\n', prob=0.1806640625, logit=17.875, token_id=341, metadata=None),
  PredictedToken(token=' and', prob=0.1806640625, logit=17.875, token_id=323, metadata=None),
  PredictedToken(token=' +', prob=0.0458984375, logit=16.5, token_id=488, metadata=None),
  PredictedToken(token=' or', prob=0.0458984375, logit=16.5, token_id=476, metadata=None)]]

In [11]:
from src.goals.eval import evaluate_mcts_paths, pretty_print_mcts_eval

evaluated_paths = evaluate_mcts_paths(
    prefix = prefix,
    paths = paths,
    unit_tests= sample.unit_tests,
    function_name=sample.function_name or "func",
)

pretty_print_mcts_eval(
    prefix = prefix,
    paths = evaluated_paths,
    function_name=sample.function_name or "func",
)

Prefix: "def is_negative(n: int) -> bool:
    return True if 0 >"

                                                   MCTS Path Evaluation Results                                                    
┏━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #    ┃ Path Text      ┃  Depth ┃  Visits ┃       Prob ┃ Verdict      ┃ Unit Tests / Errors                                      ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ " n else False │      4 │     706 │   7.07e-01 │ PASSED       │                                                          │
│      │ "              │        │         │            │              │                                                          │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 2    │ " n else False │      4 │     294 │   2.95e-01 │ PASSED       │                                                          │
│      │                │        │         │            │              │                                                          │
│      │ "              │        │         │            │              │                                                          │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 3    │ " n >"         │      2 │       0 │   9.63e-05 │ SYNTAX_ERROR │ expected 'else' after 'if' expression (<string>, line 2) │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 4    │ " else False   │      3 │       1 │   6.60e-07 │ SYNTAX_ERROR │ expected 'else' after 'if' expression (<string>, line 2) │
│      │ "              │        │         │            │              │                                                          │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 5    │ " n else "     │      3 │       0 │   3.06e-07 │ SYNTAX_ERROR │ invalid syntax (<string>, line 2)                        │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 6    │ " else False   │      3 │       0 │   2.75e-07 │ SYNTAX_ERROR │ expected 'else' after 'if' expression (<string>, line 2) │
│      │                │        │         │            │              │                                                          │
│      │ "              │        │         │            │              │                                                          │
├──────┼────────────────┼────────┼─────────┼────────────┼──────────────┼──────────────────────────────────────────────────────────┤
│ 7    │ " else:        │      2 │       0 │   1.41e-09 │ SYNTAX_ERROR │ expected 'else' after 'if' expression (<string>, line 2) │
│      │ "              │        │         │            │              │                                                          │
└──────┴────────────────┴────────┴─────────┴────────────┴──────────────┴──────────────────────────────────────────────────────────┘

Summary: 2 passed, 0 failed, 5 errors (out of 7 paths)

In [12]:
# print(search.get_stop_tokens_info())

In [13]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [14]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [15]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

{'green': 2, 'orange': 0, 'red': 3, 'total': 5}

In [16]:
search.print_tree(max_depth=None, show_colors=True)

[ROOT] visits=1006
    ├── ' n' (p=1.000, v=1003)
    │   ├── ' else' (p=1.000, v=1002)
    │   │   ├── ' False' (p=1.000, v=1001)
    │   │   │   ├── '\n' (p=0.707, v=706) [TERMINAL] (' n else False\n')
    │   │   │   └── '\n\n' (p=0.295, v=294) [TERMINAL] (' n else False\n\n')
    │   │   └── ' ' (p=0.000, v=0) (' n else ')
    │   └── ' >' (p=0.000, v=0) (' n >')
    └── ' else' (p=0.000, v=3)
        ├── ' False' (p=0.996, v=2)
        │   ├── '\n' (p=0.703, v=1) [TERMINAL] (' else False\n')
        │   └── '\n\n' (p=0.293, v=0) [TERMINAL] (' else False\n\n')
        └── ':\n' (p=0.001, v=0) [TERMINAL] (' else:\n')

In [17]:
# search.save_tree("tree.html", show_colors=True)

In [18]:
for i in range(5):
    _orig_print(i)

0
1
2
3
4


## Checking contrasting sampling

In [19]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=search.root,
    min_depth=1,
    max_depth=10,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

2026-02-17 18:45:16 src.goals.probe DEBUG    depth=1: 2 nodes, 1 positive, 1 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=2: 4 nodes, 1 positive, 1 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=3: 4 nodes, 1 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=4: 2 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=5: 0 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=6: 0 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=7: 0 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=8: 0 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=9: 0 nodes, 0 positive, 0 negative
2026-02-17 18:45:16 src.goals.probe DEBUG    depth=10: 0 nodes, 0 positive, 0 negative


Contrastive Pairs Summary

Total pairs: 2

Depths with pairs: 2

Depth range: (1, 2)

Unique positive nodes: 2

Unique negative nodes: 2

Depth 1 (1 pairs)

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Positive Path ┃ P(green) ┃ Negative Path ┃ P(red) ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ ' n'          │     1.00 │ ' else'       │   1.00 │
└───────────────┴──────────┴───────────────┴────────┘

Depth 2 (1 pairs)

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Positive Path ┃ P(green) ┃ Negative Path ┃ P(red) ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ ' n else'     │     1.00 │ ' else False' │   1.00 │
└───────────────┴──────────┴───────────────┴────────┘

In [20]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(search.root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

Nodes at depth 3: 4

' n else False...' -> OutcomeProbs(green=1.000, red=0.000, uncertain=0.000)

In [21]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")

Example Contrastive Pair at depth 1:

Positive path: ' n'

Outcome probs: OutcomeProbs(green=1.000, red=0.000, uncertain=0.000)

Node has 2 children

Negative path: ' else'

Outcome probs: OutcomeProbs(green=0.000, red=1.000, uncertain=0.000)

Node has 2 children